# Очистка истории и сравнение с решением 0,326

Проверяем два способа очистки, `median6` и `seasonal_mad`. Контроль содержит сохранённые прогнозы решения 0,326 за 18-31 октября. Все варианты оцениваются на одинаковых датах, маршрутах и горизонтах по исходным ответам.

В Kaggle подключите датасет с `cleaning_comparison_inputs.zip`, включите GPU и выполните **Run All**. Для двух веток обучаются 16 базовых CatBoost на GPU, 16 LightGBM и 8 моделей поправок CatBoost на CPU.

Каждая ветка заново рассчитывает признаки, лаги, профиль и поправки. Вес смеси и число деревьев зафиксированы. Способ очистки выбирается на 11-17 октября. Для этого периода сопоставимые прогнозы контрольной ветки отсутствуют, поэтому её сравниваем только на test. Разница с архивом включает влияние повторного обучения и изменений восстановленного кода обучения LightGBM.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import gc
import json
import zipfile
import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display

work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
out = work / 'full_pipeline_cleaning_0326'
out.mkdir(exist_ok=True)
search = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
folders = [p.parent for p in search.rglob('comparison_meta.json')]
temporary = TemporaryDirectory()
if len(folders) == 1:
    data = folders[0]
else:
    archives = list(search.rglob('cleaning_comparison_inputs.zip'))
    if len(archives) != 1:
        raise ValueError('Подключите один комплект cleaning_comparison_inputs.zip')
    data = Path(temporary.name)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.namelist():
            if Path(member).name != member:
                raise ValueError(member)
        archive.extractall(data)
meta = json.loads((data / 'comparison_meta.json').read_text())
weight = meta['weight']
start_tune = pd.Timestamp('2025-10-04')
select_from = pd.Timestamp('2025-10-11')
start_test = pd.Timestamp('2025-10-18')
keys = ['cutoff', 'horizon', 'route_id']
history = pd.read_parquet(data / 'train_solo_track.parquet').sort_values(['timestamp', 'route_id'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if (len(history) != len(times)*len(routes) or history.duplicated(['timestamp','route_id']).any()
    or not times.equals(pd.date_range(times[0],times[-1],freq='30min')) or history.isna().any().any()):
    raise ValueError('Неполная панель исходных данных')
names = ['target_1h'] + [f'status_{j}' for j in range(1,7)]
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times),len(routes)) for name in names}
del history
target = values['target_1h']

from catboost import CatBoostRegressor
from catboost.utils import get_gpu_device_count
if get_gpu_device_count() < 1:
    raise RuntimeError('В настройках Kaggle включите GPU для базового CatBoost')
raw_values = values
original_target = raw_values['target_1h']
archive_test = pd.read_parquet(data / 'validation_predictions.parquet')
TARGET = names[0]
STATUSES = names[1:]
STEP = pd.Timedelta(minutes=30)
minutes = times.hour*60+times.minute
positions = np.arange(len(times))
eval_origins = positions[(times >= start_tune) & (minutes == 630) & (positions+8 < len(times))]
if len(eval_origins) != 28 or len(routes) != 1000:
    raise ValueError('Ожидались 28 ежедневных стартов и 1000 маршрутов')

### Три версии исходных рядов
Обработка выполняется отдельно для каждого маршрута, для target_1h и всех шести статусов до расчёта любых признаков. Обе очистки ограничивают только верхние значения. Нули сохраняются.

**median6** использует верхнюю границу в шесть медиан за предыдущие 28 дней в том же получасовом слоте. Нужно 14 наблюдений и положительная медиана.

**seasonal_mad** строит сезонный прогноз для log(1+x) по медиане этого же получаса и дня недели за предыдущие четыре недели, минимум три недели. Ошибка каждого прошлого прогноза записывается в историю остатков. Для сегодняшнего слота по предыдущим 28 дням вычисляем медиану остатков и MAD - медиану абсолютных отклонений от неё. Верхняя граница равна сезонному прогнозу плюс медиана остатков плюс четыре масштаба ошибки. Масштаб равен максимуму из 1.4826 × MAD и 0.25. Нужно 14 прошлых остатков. Эти константы фиксированы до проверки. Все сезонные прогнозы, остатки и пороги используют только доступное прошлое.


In [ ]:
def clean_series(matrix, variant):
    result = matrix.copy()
    for slot in range(48):
        obs = matrix[slot::48]
        if variant == 'median6':
            median = pd.DataFrame(obs).shift(1).rolling(28,min_periods=14).median().to_numpy()
            ceiling = np.where(median > 0,6*median,np.inf)
        elif variant == 'seasonal_mad':
            logged = np.log1p(obs.astype(float))
            expected = np.full_like(logged,np.nan)
            for weekday in range(7):
                series = pd.DataFrame(logged[weekday::7])
                expected[weekday::7] = series.shift(1).rolling(4,min_periods=3).median()
            residual = logged-expected
            ceiling = np.full_like(logged,np.inf)
            for day in range(14,len(obs)):
                window = residual[max(0,day-28):day]
                enough = np.isfinite(window).sum(axis=0) >= 14
                if not enough.any():
                    continue
                previous = window[:,enough]
                center = np.nanmedian(previous,axis=0)
                mad = np.nanmedian(np.abs(previous-center),axis=0)
                upper = expected[day,enough]+center+4*np.maximum(1.4826*mad,0.25)
                ceiling[day,enough] = np.maximum(0,np.expm1(upper))
        else:
            raise ValueError(variant)
        result[slot::48] = np.minimum(obs,ceiling)
    return result

### Признаки обеих моделей
Восемь базовых CatBoost используют последние 75% разрешённых получасовых стартов. На горизонтах 5, 6, 8 добавляются сезонные признаки. Словарь задержек status_6 оценивается до начала октябрьского блока калибровки. LightGBM использует пять утренних стартов и собственный набор признаков из best. Оба типа моделей обучаются только на метках до 4 октября.


In [ ]:
def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)

def common_features(origins):
    common = {}
    for k in (0, 1, 2, 4):
        common[f"target_back_{k}"] = past(target, origins - k)
    for name, array in values.items():
        for steps in (4, 12):
            common[f"{name}_mean_{steps}"] = means[name, steps][origins]
        if name in STATUSES:
            common[f"{name}_latest"] = array[origins]
            if name in STATUSES[:3]:
                common[f"{name}_change_1h"] = array[origins] - past(array, origins - 2)
    for name, array in extra_signals.items():
        common[name] = array[origins]
    for lag in (48, 96):
        common[f"target_1h_lag_{lag}"] = past(target, origins - lag)
    common["warehouse_total_load_lag_48"] = past(extra_signals["warehouse_total_load"], origins - 48)
    common["status_6_diff_hourly_lag_48"] = past(hub_signal, origins - 48)
    common["target_vs_mean_6h"] = target[origins] - means[TARGET, 12][origins]
    recent = means[TARGET, 4][origins]
    usual = average_available([past(means[TARGET, 4], origins - day*48) for day in range(1, 8)])
    common["morning_level_ratio"] = np.divide(
        recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)
    return common

CORRELATION_COLUMNS = ["selected_status6_lag_steps", "selected_status6_correlation",
                       "selected_status6_pair_count", "status6_lag_valid",
                       "lagged_status6_signal", "lagged_status6_signal_available"]

In [ ]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def add_correlation_features(frame, h, lag_table, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}
    for name in CORRELATION_COLUMNS:
        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)
    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка признаков не сопоставлена с историей')
        selected = lag[route]
        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)
        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))
        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
        available &= np.isfinite(signal)
        for name, array in quality.items():
            frame.iloc[start:finish, column_positions[name]] = array[route]
        frame.iloc[start:finish, column_positions['lagged_status6_signal']] = np.where(available, signal, np.nan)
        frame.iloc[start:finish, column_positions['lagged_status6_signal_available']] = available.astype(np.float32)
    return frame

In [ ]:
def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins))})
    for name, matrix in common_features(origins).items():
        frame[name] = matrix.ravel()
    calendar = {'hour': dates.hour, 'minute': dates.minute, 'hour_float': dates.hour + dates.minute / 60.0, 'dayofweek': dates.dayofweek, 'is_weekend': (dates.dayofweek >= 5).astype(np.int8)}
    calendar['hour_sin'] = np.sin(2 * np.pi * calendar['hour_float'] / 24)
    calendar['hour_cos'] = np.cos(2 * np.pi * calendar['hour_float'] / 24)
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, validation_lag_table)
    for days in (1, 2, 7):
        frame[f'slot_days_back_{days}'] = past(target, slot - days * 48).ravel()
    for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric})

def add_pilot_features(frame, h, origins):
    current = target[origins]
    level = level24[origins]
    frame['level24_feature'] = level.ravel()
    frame['target_level24_logratio'] = (np.log1p(current) - np.log1p(level)).ravel()
    frame['mean2h_level24_logratio'] = (np.log1p(means[TARGET, 4][origins]) - np.log1p(level)).ravel()
    frame['profile28_minus_daily7'] = (frame.profile28 - frame.slot_daily7).astype(np.float32)
    return frame.astype({c: 'float32' for c in [*SEASONAL_FEATURES, *STATUS_FEATURES]})

SEASONAL_FEATURES = ['profile28','level24_feature','target_level24_logratio','mean2h_level24_logratio','profile28_minus_daily7']
STATUS_FEATURES = []

In [ ]:
def extend_robust(data):
    new={};floor=1/np.maximum(data.scale.to_numpy(dtype=float),1)
    for j in range(1,7):
        name=f'status_{j}';level=np.maximum(data[f'long_{name}_mean48'].to_numpy(dtype=float),floor)
        week=np.maximum(data[f'long_{name}_mean336'].to_numpy(dtype=float),floor)
        for suffix,key in [('current',f'{name}_lag0'),('mean4',f'{name}_mean4'),('mean12',f'{name}_mean12'),('delta4',f'{name}_delta4')]:
            new[f'robust_s{j}_{suffix}']=np.clip(data[key].to_numpy()/level,-100,100).astype(np.float32)
        new[f'robust_s{j}_level_change']=np.clip(level/week,0,100).astype(np.float32)
        raw=data[f'{name}_lag0'].to_numpy(dtype=float)*data.scale.to_numpy()
        denominator=pd.Series(raw).groupby([data.cutoff,data.horizon],sort=False).transform('sum').to_numpy()
        new[f'robust_s{j}_route_share']=np.divide(raw,np.maximum(denominator,1)).astype(np.float32)
    for window in [4,12]:
        new[f'robust_target_vs{window}']=np.clip(data.target_lag_0.to_numpy()/np.maximum(data[f'target_mean_{window}'].to_numpy(),floor),0,100).astype(np.float32)
    return pd.concat([data,pd.DataFrame(new,index=data.index)],axis=1)

In [ ]:
def features(origins,h,profile_history,daily):
    T = len(times)
    R = len(routes)
    def past(a,index):
        index=np.asarray(index);out=a[np.clip(index,0,T-1)].astype(np.float32,copy=True)
        out[(index<0)|(index>=T)]=np.nan
        return out
    scale=np.maximum(daily[origins],1)
    common={}
    for lag in (*range(9),12,24,48,96,336):common[f'target_lag_{lag}']=past(target,origins-lag)/scale
    for window in (4,12,48):common[f'target_mean_{window}']=(daily if window == 48 else means['target_1h',window])[origins]/scale
    for window in (4,12):common[f'target_std_{window}']=deviations[window][origins]/scale
    for name in names[1:]:
        a=values[name]
        for lag in (0,1,2):common[f'{name}_lag{lag}']=past(a,origins-lag)/scale
        for window in (4,12):common[f'{name}_mean{window}']=means[name,window][origins]/scale
        for lag in (2,4):common[f'{name}_delta{lag}']=(a[origins]-past(a,origins-lag))/scale
    common['log_current_level']=np.log1p(scale)
    for name,a in values.items():
        total=a.sum(axis=1,dtype=np.float64)
        netlevel=pd.Series(total).rolling(48,min_periods=1).mean().to_numpy()
        net12=pd.Series(total).rolling(12,min_periods=1).mean().to_numpy()
        den=np.maximum(netlevel[origins],1)
        for suffix,x in [('level',total[origins]/den),('mean12',net12[origins]/den),
                         ('delta2',(total[origins]-total[origins-2])/den)]:
            common[f'network_{name}_{suffix}']=np.broadcast_to(x[:,None],scale.shape).astype(np.float32)
    common['route_network_share']=target[origins]/np.maximum(target[origins].sum(axis=1,keepdims=True),1)
    slot=origins+h;dates=times[origins]+pd.Timedelta(minutes=30*h)
    arrays={name:a.ravel() for name,a in common.items()}
    for day in (1,2,3,7,14,21,28):
        a=past(target,slot-day*48);arrays[f'slot_day{day}']=(a/scale).ravel()
    for day in (1,7):
        for offset in (-2,-1,1,2):arrays[f'slot_day{day}_offset{offset}']=(past(target,slot-day*48+offset)/scale).ravel()
    stack=np.stack([past(target,slot-day*48) for day in range(1,8)])
    arrays['slot_daily7']=(np.nanmean(stack,axis=0)/scale).ravel()
    stack=np.stack([past(target,slot-day*48) for day in (7,14,21,28)])
    with np.errstate(invalid='ignore'):weekly=np.nanmean(stack,axis=0)
    arrays['slot_weekly4']=(weekly/scale).ravel()
    profile_sum=np.zeros_like(scale,dtype=np.float64);count=np.zeros_like(scale)
    for day in range(1,29):
        a=past(profile_history,slot-day*48)*scale/np.maximum(past(daily,origins-day*48),1)
        valid=np.isfinite(a);profile_sum+=np.nan_to_num(a);count+=valid
    profile=np.divide(profile_sum,count,out=np.zeros_like(profile_sum),where=count>0).astype(np.float32)
    arrays['profile_norm']=(profile/scale).ravel()
    arrays['horizon']=np.full(len(origins)*R,h,dtype=np.int8)
    arrays['dayofweek']=np.repeat(dates.dayofweek,R).astype(np.int8)
    arrays['origin_minute']=np.repeat(times[origins].hour*60+times[origins].minute,R).astype(np.int16)
    arrays['future_hour']=np.repeat(dates.hour+dates.minute/60,R).astype(np.float32)
    frame=pd.DataFrame(arrays)
    frame['cutoff']=times[origins].repeat(R);frame['timestamp']=dates.repeat(R)
    frame['route_id']=np.tile(routes,len(origins)).astype(np.int16)
    frame['scale']=scale.ravel();frame['profile28']=profile.ravel()
    frame['y']=past(target,slot).ravel()
    extra = {}
    for name in names[1:]:
        for lag in (4,8,12,24,48,96):
            extra[f'long_{name}_lag{lag}'] = (past(values[name],origins-lag)/scale).ravel()
        for window, series in [(48,means[name,48]),(336,long_week[name])]:
            extra[f'long_{name}_mean{window}'] = (series[origins]/scale).ravel()
    return extend_robust(pd.concat([frame,pd.DataFrame(extra)],axis=1))

In [ ]:
def prepare_history(branch):
    global values, target, means, level24, extra_signals, hub_signal, validation_lag_table, long_week, deviations
    values = branch
    target = values[TARGET]
    means = {(name, steps): rolling(array, steps)
             for name, array in values.items() for steps in (4, 12)}
    level24 = rolling(target, 48)
    positions = np.arange(len(times))
    extra_signals = {}
    for name in STATUSES[:3]:
        extra_signals[name + "_hourly"] = values[name] + past(values[name], positions - 1)
    extra_signals["warehouse_total_load"] = sum(extra_signals[name + "_hourly"] for name in STATUSES[:3])
    extra_signals["warehouse_processing_rate"] = (
        extra_signals["status_3_hourly"] / (extra_signals["status_1_hourly"] + np.float32(1e-5)))
    for name in STATUSES[3:]:
        extra_signals[name + "_diff_hourly"] = values[name] - past(values[name], positions - 2)
    extra_signals["hub_buffer"] = values["status_5"] - values["status_4"]
    extra_signals["hub_gate_bottleneck"] = (
        extra_signals["status_4_diff_hourly"] - extra_signals["status_6_diff_hourly"])
    extra_signals["target_acceleration"] = past(target, positions - 1) - past(target, positions - 2)
    hub_signal = extra_signals["status_6_diff_hourly"]
    
    means.update({(name,48):rolling(a,48) for name,a in values.items()})
    long_week = {name:rolling(values[name],336) for name in STATUSES}
    deviations = {w:pd.DataFrame(target).rolling(w,min_periods=2).std().to_numpy(dtype=np.float32) for w in (4,12)}
    validation_lag_table = calculate_dynamic_shoulder(target,hub_signal,times,routes,fit_before=start_tune)

In [ ]:
cat_columns = ['route_id', 'target_back_0', 'target_back_1', 'target_back_2', 'target_back_4', 'target_1h_mean_4', 'target_1h_mean_12', 'status_1_mean_4', 'status_1_mean_12', 'status_1_latest', 'status_1_change_1h', 'status_2_mean_4', 'status_2_mean_12', 'status_2_latest', 'status_2_change_1h', 'status_3_mean_4', 'status_3_mean_12', 'status_3_latest', 'status_3_change_1h', 'status_4_mean_4', 'status_4_mean_12', 'status_4_latest', 'status_5_mean_4', 'status_5_mean_12', 'status_5_latest', 'status_6_mean_4', 'status_6_mean_12', 'status_6_latest', 'status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate', 'status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck', 'target_acceleration', 'target_1h_lag_48', 'target_1h_lag_96', 'warehouse_total_load_lag_48', 'status_6_diff_hourly_lag_48', 'target_vs_mean_6h', 'morning_level_ratio', 'hour', 'minute', 'hour_float', 'dayofweek', 'is_weekend', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available', 'slot_days_back_1', 'slot_days_back_2', 'slot_days_back_7', 'slot_daily7', 'slot_weekly4']
cat_trees = [1500, 1500, 1500, 1500, 1499, 1500, 1497, 1500]
anchor_columns = ['target_back_0','target_back_1','target_back_2','target_back_4',
                  'target_1h_mean_4','target_1h_mean_12','level24_feature','target_1h_lag_48','target_1h_lag_96',
                  'target_vs_mean_6h','morning_level_ratio','warehouse_processing_rate','target_acceleration']
anchor_columns += [f'status_{j}_{suffix}' for j in range(1,7) for suffix in ('latest','mean_4','mean_12')]

def make_cat_frame(origins,h):
    return add_pilot_features(horizon_dataset(h,origins),h,origins)

def metric(actual,predicted):
    y = np.asarray(actual,dtype=float)
    p = np.asarray(predicted,dtype=float)
    if y.shape != p.shape or not np.isfinite(y).all() or not np.isfinite(p).all() or y.sum() <= 0:
        raise ValueError('Неверные данные для метрики')
    error = p-y
    wape = np.abs(error).sum()/y.sum()
    bias = error.sum()/y.sum()
    return dict(rows=len(y),WAPE=wape,relative_bias=bias,score=wape+abs(bias))

log = []
def report(branch,stage,h,split,y,p):
    log.append(dict(branch=branch,stage=stage,h=h,split=split,**metric(y,p)))

def show_fit():
    display(pd.DataFrame(log[-2:]))
    pd.DataFrame(log).to_csv(out/'metrics_after_fit.csv',index=False)

def base_predictions(branch,h):
    eligible = positions[times+h*STEP < start_tune]
    cat_origins = eligible[len(eligible)//4:]
    columns = cat_columns + (SEASONAL_FEATURES if h in (5,6,8) else [])
    chunks = []
    for batch in range(0,len(cat_origins),32):
        chunks.append(make_cat_frame(cat_origins[batch:batch+32],h)[columns])
    train = pd.concat(chunks,ignore_index=True)
    del chunks
    cat = CatBoostRegressor(iterations=cat_trees[h-1],depth=7,learning_rate=.05,loss_function='MAE',
                            task_type='GPU',devices='0',cat_features=['route_id'],random_seed=42,
                            allow_writing_files=False,verbose=250,thread_count=4)
    cat.fit(train,target[cat_origins+h].ravel()/100000)
    model_dir = out/'models'/branch
    cat.save_model(str(model_dir/f'catboost_h{h}.cbm'))
    fit_prediction = np.maximum(cat.predict(train)*100000,0)
    report(branch,'CatBoost',h,'train',original_target[cat_origins+h].ravel(),fit_prediction)
    del train,fit_prediction
    f = make_cat_frame(eval_origins,h)
    frame = f[['cutoff','route_id','timestamp','profile28',*anchor_columns]].copy()
    frame['horizon'] = h
    frame['y'] = original_target[eval_origins+h].ravel()
    frame['fit_y'] = target[eval_origins+h].ravel()
    frame['model_prediction'] = np.maximum(cat.predict(f[columns])*100000,0)
    test = frame.cutoff >= start_test
    report(branch,'CatBoost',h,'test',frame.loc[test,'y'],frame.loc[test,'model_prediction'])
    show_fit()
    frame['raw_blend'] = (frame.model_prediction if h == 1 else .7*frame.model_prediction+.3*frame.profile28)
    del cat,f
    origins = eligible[(eligible >= 336) & np.isin(minutes[eligible],[570,600,630,660,690])]
    train = features(origins,h,target,level24)
    cfg = meta[str(h)]
    cols = cfg['columns']
    model = lgb.LGBMRegressor(**dict(cfg['params'],n_estimators=cfg['trees'],n_jobs=4))
    scale = train.scale.to_numpy(dtype=float)
    model.fit(train[cols],target[origins+h].ravel()/scale,sample_weight=scale/scale.mean())
    model.booster_.save_model(str(model_dir/f'lightgbm_h{h}.txt'))
    fit_prediction = np.maximum(model.predict(train[cols])*scale,0)
    report(branch,'LightGBM',h,'train',original_target[origins+h].ravel(),fit_prediction)
    del train,fit_prediction
    evaluation = features(eval_origins,h,target,level24)
    if not frame[keys].reset_index(drop=True).equals(evaluation[keys].astype(frame[keys].dtypes).reset_index(drop=True)):
        raise ValueError('Расходятся ключи между базовыми моделями')
    frame['shape'] = np.maximum(model.predict(evaluation[cols])*evaluation.scale.to_numpy(dtype=float),0)
    report(branch,'LightGBM',h,'test',frame.loc[test,'y'],frame.loc[test,'shape'])
    show_fit()
    return frame

### Поправки и подбор до test
Базовые прогнозы 4-17 октября построены вне обучения базовых моделей. Для выбора ветки две модели остатка обучаются на 4-10 октября и прогнозируют 11-17 октября. Для проверки test переобучаем обе поправки на всех 14 днях 4-17 октября, как в рецепте 0.328. На каждом старте поправка среднего смещения использует только уже известные ошибки, с полупериодом семь дней и запасом четыре часа. В очищенных ветках наблюдаемая история для этой поправки тоже проходит фильтр. Фактические ответы для метрик всегда сырые.

На первом этапе после каждого fit поправки выводится диагностический test с семидневным обучением. Выбор ветки в коде задан только по средней дневной метрике tuning. Итоговый train ансамбля относится к периоду обучения моделей поправки 4-17 октября.


In [ ]:
def residual_features(frame):
    cols = ['route_id','horizon','raw_blend','model_prediction','profile28',*anchor_columns]
    X = frame[cols].copy()
    X['dayofweek'] = frame.cutoff.dt.dayofweek
    X['route_id'] = X.route_id.astype(str)
    X['horizon'] = X.horizon.astype(str)
    for name in ['model_prediction','profile28','target_back_0','target_1h_mean_4','target_1h_mean_12',
                 'level24_feature','target_1h_lag_48','target_1h_lag_96']:
        X[f'{name}_minus_base'] = X[name]-X.raw_blend
    return X

def causal_offsets(frame,initial,unshifted,fit_mask):
    errors = frame.fit_y.to_numpy()-unshifted
    offsets = np.full(len(frame),initial)
    for cutoff in frame.loc[~fit_mask,'cutoff'].drop_duplicates():
        known = frame.timestamp < cutoff-pd.Timedelta(hours=4)
        if frame.loc[known,'cutoff'].nunique() >= 3:
            ages = (cutoff-frame.loc[known,'cutoff']).dt.total_seconds().to_numpy()/86400
            offsets[frame.cutoff.eq(cutoff)] = np.average(errors[known],weights=np.exp2(-ages/7))
    return offsets

def finish_pipeline(frame,branch,fit_before):
    fit = frame.cutoff < fit_before
    test = frame.cutoff >= start_test
    X = residual_features(frame)
    corrected = []
    for depth,trees,l2 in [(4,450,30),(5,900,50)]:
        model = CatBoostRegressor(iterations=trees,depth=depth,learning_rate=.03,l2_leaf_reg=l2,
                                  loss_function='MAE',cat_features=['route_id','horizon'],random_seed=42,
                                  thread_count=4,allow_writing_files=False,verbose=False)
        model.fit(X.loc[fit],(frame.loc[fit,'fit_y']-frame.loc[fit,'raw_blend'])/100000)
        unshifted = frame.raw_blend.to_numpy()+model.predict(X)*100000
        initial = np.mean(frame.loc[fit,'fit_y']-unshifted[fit])
        folder = out/'models'/branch/f'corrections_before_{fit_before:%Y%m%d}'
        folder.mkdir(parents=True,exist_ok=True)
        model.save_model(str(folder/f'depth{depth}.cbm'))
        state = frame[keys+['timestamp']].copy()
        state['residual'] = frame.fit_y.to_numpy()-unshifted
        state['initial_offset'] = float(initial)
        state['used_for_fit'] = fit.to_numpy()
        state.to_parquet(folder/f'depth{depth}_history.parquet',index=False)
        pred = np.maximum(unshifted+causal_offsets(frame,initial,unshifted,fit),0)
        stage = f'residual_depth{depth}_fit_before_{fit_before:%m%d}'
        report(branch,stage,'all','train',frame.loc[fit,'y'],pred[fit])
        report(branch,stage,'all','test',frame.loc[test,'y'],pred[test])
        show_fit()
        corrected.append(pred)
    p = frame[keys+['y','shape']].copy()
    p['volume_model'] = (corrected[0]+corrected[1])/2
    totals = p.groupby(['cutoff','horizon'])[['volume_model','shape']].transform('sum')
    if totals['shape'].le(0).any():
        raise ValueError('Нулевая сумма LightGBM')
    p['prediction'] = (1-weight)*p.volume_model+weight*p['shape']*totals.volume_model/totals['shape']
    p['branch'] = branch
    return p

In [ ]:
control = archive_test[keys+['y','session_prediction']].rename(columns={'session_prediction':'prediction'})
control = control.assign(branch='raw',split='test')
if len(control) != 112000 or control.duplicated(keys).any():
    raise ValueError('Архив должен содержать 112000 уникальных строк test')
predictions = [control]
counts = []
for branch in ['median6','seasonal_mad']:
    print('Подготовка ветки',branch,flush=True)
    panel = {name:clean_series(a,branch) for name,a in raw_values.items()}
    for name in names:
        for split,mask in [('train',times < start_tune),('tuning',(times >= start_tune)&(times < start_test)),
                           ('test',(times >= start_test)&(times < times[-1].normalize()))]:
            changed = np.count_nonzero(panel[name][mask] != raw_values[name][mask])
            counts.append(dict(branch=branch,field=name,split=split,changed=changed,rows=int(mask.sum()*len(routes))))
    pd.DataFrame(counts).to_csv(out/'cleaning_counts.csv',index=False)
    prepare_history(panel)
    folder = out/'models'/branch
    folder.mkdir(parents=True,exist_ok=True)
    validation_lag_table.to_csv(folder/'status6_lags.csv',index=False)
    parts = []
    for h in range(1,9):
        parts.append(base_predictions(branch,h))
        gc.collect()
    frame = pd.concat(parts,ignore_index=True).sort_values(keys).reset_index(drop=True)
    if frame.duplicated(keys).any() or len(frame) != 224000:
        raise ValueError('Неполная панель сравнения')
    tune_run = finish_pipeline(frame,branch,select_from)
    tuned = tune_run[(tune_run.cutoff >= select_from)&(tune_run.cutoff < start_test)].copy()
    tuned['split'] = 'tuning'
    test_run = finish_pipeline(frame,branch,start_test)
    test_run['split'] = np.where(test_run.cutoff < start_test,'train','test')
    result = pd.concat([tuned,test_run],ignore_index=True)
    matched = test_run.loc[test_run.split.eq('test'),keys+['y']].merge(
        control[keys+['y']],on=keys,how='outer',validate='one_to_one',indicator=True,suffixes=('_new','_raw'))
    if not matched['_merge'].eq('both').all() or not np.array_equal(matched.y_new,matched.y_raw):
        raise ValueError('Ключи или факты test расходятся с архивным контролем')
    result.to_parquet(out/f'predictions_{branch}.parquet',index=False)
    predictions.append(result)
    del panel,parts,frame,tune_run,test_run
    gc.collect()

In [ ]:
paired = pd.concat(predictions,ignore_index=True)
def summary(frame):
    daily = [metric(day.y,day.prediction)['score'] for _,day in frame.groupby('cutoff')]
    return dict(**metric(frame.y,frame.prediction),mean_daily_score=np.mean(daily))
report_rows = [dict(branch=branch,split=split,**summary(d))
               for (branch,split),d in paired.groupby(['branch','split'],sort=False)]
scores = pd.DataFrame(report_rows)
raw_scores = scores[scores.branch.eq('raw')].set_index('split').score
scores['delta_vs_raw'] = scores.score-scores.split.map(raw_scores)
winner = scores[scores.split.eq('tuning')].sort_values('mean_daily_score',kind='stable').iloc[0].branch
scores.to_csv(out/'pipeline_scores.csv',index=False)
print('Лучший из двух способов очистки по tuning:',winner)
display(scores)
for key in ['cutoff','horizon']:
    details = pd.DataFrame([dict(branch=b,**{key:v},**metric(d.y,d.prediction))
        for (b,v),d in paired[paired.split.eq('test')].groupby(['branch',key])])
    details.to_csv(out/f'test_by_{key}.csv',index=False)
print('Оценка test независима от выбора в этом запуске и получена до финального переобучения.')
print('Период 18–31 октября ранее просматривался. Сабмит в этом эксперименте не собирается.')

## Сохранение результатов

Последняя ячейка создаёт `full_pipeline_cleaning_0326.zip` с отчётами, прогнозами, моделями, настройками и входными данными. Модели в архиве относятся к оценке до финального переобучения.


In [ ]:
from importlib.metadata import version
from IPython.display import FileLink

def export_results():
    files = {}
    for name in ['pipeline_scores.csv','metrics_after_fit.csv','cleaning_counts.csv',
                 'test_by_cutoff.csv','test_by_horizon.csv']:
        files[name] = out/name
    for name in ['train_solo_track.parquet','test_solo_track.parquet','comparison_meta.json',
                 'validation_predictions.parquet']:
        files[f'inputs/{name}'] = data/name
    for branch in ['median6','seasonal_mad']:
        name = f'predictions_{branch}.parquet'
        files[name] = out/name
        folder = Path('models')/branch
        models = [folder/'status6_lags.csv']
        for h in range(1,9):
            models.extend([folder/f'catboost_h{h}.cbm',folder/f'lightgbm_h{h}.txt'])
        for boundary in [select_from,start_test]:
            phase = folder/f'corrections_before_{boundary:%Y%m%d}'
            for depth in [4,5]:
                models.extend([phase/f'depth{depth}.cbm',phase/f'depth{depth}_history.parquet'])
        files.update({path.as_posix():out/path for path in models})
    for name,path in files.items():
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(f'Отсутствует файл результата: {name}')
    manifest = dict(selected_on_tuning=winner,shape_weight=weight,
                    evaluation_before_final_refit=True,test_period=['2025-10-18','2025-10-31'],
                    target='target_1h',cleaning_fields=names,median6=dict(days=28,min_days=14,multiplier=6),
                    seasonal_mad=dict(transform='log1p',weeks=4,min_weeks=3,residual_days=28,
                                      min_residuals=14,multiplier=4,mad_scale=1.4826,scale_floor=0.25),
                    catboost_features=cat_columns,seasonal_features=SEASONAL_FEATURES,
                    seasonal_horizons=[5,6,8],catboost_trees=cat_trees,
                    versions={pkg:version(pkg) for pkg in ['catboost','lightgbm','numpy','pandas','pyarrow']},
                    files=sorted(files))
    destination = work/'full_pipeline_cleaning_0326.zip'
    with TemporaryDirectory(dir=work) as staging:
        candidate = Path(staging)/destination.name
        with zipfile.ZipFile(candidate,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=2) as archive:
            for name,path in files.items():
                archive.write(path,name)
            archive.writestr('manifest.json',json.dumps(manifest,ensure_ascii=False,indent=2))
        with zipfile.ZipFile(candidate) as archive:
            broken = archive.testzip()
            if broken is not None:
                raise ValueError(f'Ошибка проверки ZIP: {broken}')
        candidate.replace(destination)
    return destination

result_zip = export_results()
print(f'{result_zip} ({result_zip.stat().st_size/1024**2:.1f} МиБ)')
display(FileLink(result_zip.name))
